# Reproduction check -- seed 2 only

Re-runs **only seed 2**, with every other setting held fixed, and compares the fresh
predictions against the ones stored by the 20-seed sweep in
`output/domain-shift-tng100-20seeds/predictions/`.

The setup, data-prep and training cells below are lifted verbatim from
`domain_shift_tng100_20seeds.ipynb` (read out of that notebook when this one was
generated), so the configuration cannot silently drift between the two. The only
differences are `SEEDS = [2]` and a separate `OUTPUT_DIR`.

**What a pass means.** The sweep already verified that the same seed reproduces itself
*within a single process*. This is the stronger check: a completely fresh process, new
CUDA context, graphs rebuilt from scratch. If predictions come back bitwise identical,
then `seed=2` pins the result across processes and the recorded seeds are portable, not
just session-local.

**What a fail would mean.** Bitwise differences would say some state outside the seed is
leaking into training -- library version, GPU context, or an unseeded RNG consumer -- and
the recorded seeds would only be labels. The comparison prints max |diff| either way, so
a near-miss (~1e-7, floating-point noise) is distinguishable from a real divergence.


In [ ]:
# NOTE: CUBLAS_WORKSPACE_CONFIG must be set BEFORE torch initialises CUDA,
# otherwise torch.use_deterministic_algorithms() raises on some cuBLAS ops.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm, t as student_t

torch.use_deterministic_algorithms(True, warn_only=True)

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / "scripts").exists() and (candidate / "data").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate repository root containing 'scripts' and 'data'.")

SCRIPTS_DIR = REPO_ROOT / "scripts"
DM_ROOT = REPO_ROOT / "data"
SHIFTKIT_DIR = REPO_ROOT / "ShiftKit"

for path in [str(REPO_ROOT), str(SCRIPTS_DIR), str(DM_ROOT), str(SHIFTKIT_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from process_data import DataProcessor, sample_pyg_subgraph
from process_tng_data import TNGDataProcessor

REFERENCE_DIR = REPO_ROOT / "output" / "domain-shift-tng100-20seeds"
OUTPUT_DIR = REPO_ROOT / "output" / "domain-shift-tng100-seed2-repro"
CKPT_DIR = OUTPUT_DIR / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DIRECTION = "tng_to_fire"
SRC_NAME, TGT_NAME = "TNG100", "FIREbox"

# 20 recorded seeds. Explicit list (not a count) so it is unambiguous what was run.
SEEDS = [2]   # reproduction run: this notebook trains ONLY seed 2

# Training config -- matches domain_shift_tng100_4methods.ipynb
MODEL_CONFIG = {"hidden_channels": 64, "num_layers": 2, "model_name": "SAGE"}
EPOCHS = 200
WARMUP = 10
LR = 5e-3
POTENTIAL_TEMPERATURE = 0.5
OT_EMA_MOMENTUM = 0.9

METHOD_SPECS = [
    dict(label="No-DA",                 kind="sourceonly"),
    dict(label="SIDDA",                 kind="sidda", use_potentials=False, weight_ot=False),
    dict(label="SIDDA + OT-reweight",   kind="sidda", use_potentials=False, weight_ot=True),
    dict(label="SIDDA + Loss-reweight", kind="sidda", use_potentials=True,  weight_ot=False),
]
ALL_METHOD_ORDER = [m["label"] for m in METHOD_SPECS]

METHOD_COLORS = {
    "No-DA": "#4C72B0",
    "SIDDA": "#DD8452",
    "SIDDA + OT-reweight": "#55A868",
    "SIDDA + Loss-reweight": "#C44E52",
}

# Data split / graph construction -- fixed, shared by every run.
CONFIG = {
    "fire_path": DM_ROOT / "firebox_data" / "FIREbox_z=0.txt",
    "tng_path": DM_ROOT / "tng-data" / "TNG100" / "subhalos_99.parquet",
    "r": 1,
    "test_size": 0.1,
    "val_size": 0.1,
    "standardize": True,
    "stratify_bins": 10,
    "random_state": 42,
    "upper_mass": 12,
}
GRAPH_KWARGS = dict(
    r=CONFIG["r"], test_size=CONFIG["test_size"], val_size=CONFIG["val_size"],
    standardize=CONFIG["standardize"], stratify_bins=CONFIG["stratify_bins"],
    random_state=CONFIG["random_state"], upper_mass=CONFIG["upper_mass"],
)

METRICS = ["src_rmse", "src_r2", "src_chi2", "tgt_rmse", "tgt_r2", "tgt_chi2"]
DDOF = 1  # sample std everywhere -- see the stats note below


def ref_npz_path(label, seed):
    """Path to the 20-seed sweep's stored prediction file for this run."""
    stem = label.replace(" ", "_")
    return REFERENCE_DIR / "predictions" / f"{stem}_seed{seed}_predictions.npz"


def npz_path(label, seed):
    return CKPT_DIR / f"{label.replace(' ', '_')}_seed{seed}_predictions.npz"


def compute_metrics(true, mean, std):
    """RMSE, R^2 and reduced chi^2 from a single run's predictions."""
    resid = mean - true
    mse = float(np.mean(resid ** 2))
    ss_res = float(np.sum(resid ** 2))
    ss_tot = float(np.sum((true - true.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    pulls = resid / std
    return {"rmse": mse ** 0.5, "r2": r2, "chi2": float(np.mean(pulls ** 2))}


print(f"Repo:            {REPO_ROOT}")
print(f"Output:          {OUTPUT_DIR}")
print(f"Seeds:           {SEEDS}")
print(f"Config:          EPOCHS={EPOCHS}  temp={POTENTIAL_TEMPERATURE}  LR={LR}  warmup={WARMUP}")
print(f"CUDA available:  {torch.cuda.is_available()}"
      + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Deterministic:   {torch.are_deterministic_algorithms_enabled()}"
      f"  (CUBLAS_WORKSPACE_CONFIG={os.environ.get('CUBLAS_WORKSPACE_CONFIG')})")


## Rebuild the graphs (identical construction and split)

In [ ]:
USE_FULL_TNG_GRAPH = True  # False reproduces the size-matched (1,035-node) TNG graph

fire_proc = DataProcessor(file_path=str(CONFIG["fire_path"]), subhalos="both")
fire_data = fire_proc.create_graph_data(**GRAPH_KWARGS)

tng_proc = TNGDataProcessor(file_path=str(CONFIG["tng_path"]), subhalos="both")

MAX_TNG_ROWS_FOR_GRAPH = 7500
if len(tng_proc.df_filtered) > MAX_TNG_ROWS_FOR_GRAPH:
    rng_pre = np.random.default_rng(CONFIG["random_state"])
    keep_idx = np.sort(rng_pre.choice(len(tng_proc.df_filtered), size=MAX_TNG_ROWS_FOR_GRAPH, replace=False))
    print(f"Pre-subsampling TNG100 catalog: {len(tng_proc.df_filtered):,} -> {MAX_TNG_ROWS_FOR_GRAPH:,} rows")
    tng_proc.df_filtered = tng_proc.df_filtered.iloc[keep_idx].reset_index(drop=True)

tng_data = tng_proc.create_graph_data(**GRAPH_KWARGS)

FIRE_MSTAR_COL = "lg_Mstar_<Rhalo"
TNG_MSTAR_COL = "stellar_mass"
fdf = fire_proc.df_filtered
tdf_full = tng_proc.df_filtered
overlap_lo = tdf_full[TNG_MSTAR_COL].values.astype(float).min()
overlap_hi = fdf[FIRE_MSTAR_COL].values.astype(float).max()
print(f"Overlap range: [{overlap_lo:.2f}, {overlap_hi:.2f}] dex in lg_Mstar")

from torch_geometric.data import Data


def induced_subgraph_by_mask(data, keep_mask):
    '''Induced subgraph over nodes where keep_mask is True (split masks carried over).'''
    keep_idx = keep_mask.nonzero(as_tuple=False).view(-1)
    index_map = {int(old): new for new, old in enumerate(keep_idx.tolist())}
    keep_set = set(keep_idx.tolist())

    edge_index = data.edge_index
    src, dst = edge_index[0].tolist(), edge_index[1].tolist()
    edge_mask = torch.tensor([s in keep_set and d in keep_set for s, d in zip(src, dst)], dtype=torch.bool)
    kept_edges = edge_index[:, edge_mask]
    remapped = torch.stack([
        torch.tensor([index_map[int(s)] for s in kept_edges[0].tolist()], dtype=torch.long),
        torch.tensor([index_map[int(d)] for d in kept_edges[1].tolist()], dtype=torch.long),
    ])

    sub = Data(
        x=data.x[keep_idx],
        edge_index=remapped,
        y=data.y[keep_idx],
        pos=data.pos[keep_idx] if getattr(data, "pos", None) is not None else None,
    )
    for split in ("train_mask", "val_mask", "test_mask"):
        if hasattr(data, split):
            setattr(sub, split, getattr(data, split)[keep_idx])
    return sub


fire_keep = torch.tensor((fdf[FIRE_MSTAR_COL].values >= overlap_lo) & (fdf[FIRE_MSTAR_COL].values <= overlap_hi))
tng_keep = torch.tensor((tdf_full[TNG_MSTAR_COL].values >= overlap_lo) & (tdf_full[TNG_MSTAR_COL].values <= overlap_hi))

fire_overlap_graph = induced_subgraph_by_mask(fire_data, fire_keep)
tng_overlap_full_graph = induced_subgraph_by_mask(tng_data, tng_keep)
tng_overlap_matched_graph = sample_pyg_subgraph(
    tng_overlap_full_graph, num_nodes_sample=fire_overlap_graph.num_nodes,
    random_state=CONFIG["random_state"],
)
tng_graph = tng_overlap_full_graph if USE_FULL_TNG_GRAPH else tng_overlap_matched_graph
tng_graph_label = "TNG100 (overlap, full)" if USE_FULL_TNG_GRAPH else "TNG100 (overlap, size-matched)"

for name, d in [("FIREbox (overlap)", fire_overlap_graph), (tng_graph_label, tng_graph)]:
    print(f"{name:32s} nodes={d.num_nodes:,}  edges={d.num_edges:,}  "
          f"train={int(d.train_mask.sum())}  val={int(d.val_mask.sum())}  test={int(d.test_mask.sum())}")

# The split must be a genuine partition, and must come from DataProcessor's stratified
# split rather than any fallback -- fail loudly if not.
for name, d in [("FIREbox", fire_overlap_graph), (tng_graph_label, tng_graph)]:
    masks = [d.train_mask, d.val_mask, d.test_mask]
    assert all(m is not None and bool(m.any()) for m in masks), f"{name}: missing/empty split mask"
    assert sum(int(m.sum()) for m in masks) == d.num_nodes, f"{name}: masks do not partition the nodes"
    assert not bool((d.train_mask & d.val_mask).any() or (d.train_mask & d.test_mask).any()
                    or (d.val_mask & d.test_mask).any()), f"{name}: split masks overlap"
    frac = int(d.test_mask.sum()) / d.num_nodes
    assert abs(frac - CONFIG["test_size"]) < 0.03, (
        f"{name}: test fraction {frac:.3f} != CONFIG['test_size'] {CONFIG['test_size']}")
print("split masks verified: stratified partition, shared by every run")

DIRECTIONS = {
    DIRECTION: dict(src=tng_graph, tgt=fire_overlap_graph, src_name=SRC_NAME, tgt_name=TGT_NAME),
}


## Train seed 2 only -- same training cell as the sweep

In [ ]:
import time
import csv

from shiftkit import (
    GNN, DataManager,
    SourceOnlyGaussianRegressionTrainer, SIDDAGaussianRegressionTrainer,
)

progress_log = OUTPUT_DIR / "progress.log"
partial_csv = OUTPUT_DIR / "per_seed_results_partial.csv"


def log_progress(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(progress_log, "a") as f:
        f.write(line + "\n")


def build_trainer(spec, model, train_src, train_tgt):
    if spec["kind"] == "sourceonly":
        return SourceOnlyGaussianRegressionTrainer(model, train_src, train_tgt, lr=LR)
    if spec["kind"] == "sidda":
        return SIDDAGaussianRegressionTrainer(
            model, train_src, train_tgt, lr=LR, warmup_epochs=WARMUP,
            use_potentials=spec["use_potentials"], weight_ot=spec["weight_ot"],
            potential_temperature=POTENTIAL_TEMPERATURE, ot_ema_momentum=OT_EMA_MOMENTUM,
        )
    raise ValueError(spec["kind"])


spec = DIRECTIONS[DIRECTION]
src, tgt = spec["src"], spec["tgt"]
dm = DataManager(batch_size=1, num_workers=0)
train_src, train_tgt = dm.load("pyg_domains", train=True, source=src, target=tgt)
test_src, test_tgt = dm.load("pyg_domains", train=False, source=src, target=tgt)

if not partial_csv.exists():
    with open(partial_csv, "w", newline="") as f:
        csv.writer(f).writerow(["method", "seed"] + METRICS + ["elapsed_s"])

t_sweep = time.time()
for method_spec in METHOD_SPECS:
    label = method_spec["label"]
    for seed in SEEDS:
        t0 = time.time()

        # Reseed immediately before construction: makes (method, seed) a deterministic
        # function of `seed` alone, not of the accumulated RNG state.
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        model = GNN(
            src, MODEL_CONFIG["model_name"],
            hidden_channels=MODEL_CONFIG["hidden_channels"],
            num_layers=MODEL_CONFIG["num_layers"],
            regress=True, pool="none", predict_var=True,
        )
        trainer = build_trainer(method_spec, model, train_src, train_tgt)
        trainer.fit(epochs=EPOCHS)
        elapsed = time.time() - t0

        true_src, mean_src, std_src = trainer.predict(test_src)
        true_tgt, mean_tgt, std_tgt = trainer.predict(test_tgt)

        np.savez(
            npz_path(label, seed),
            true_src=true_src, mean_src=mean_src, std_src=std_src,
            true_tgt=true_tgt, mean_tgt=mean_tgt, std_tgt=std_tgt,
            seed=np.array([seed]),
        )

        ms = compute_metrics(true_src, mean_src, std_src)
        mt = compute_metrics(true_tgt, mean_tgt, std_tgt)
        with open(partial_csv, "a", newline="") as f:
            csv.writer(f).writerow([
                label, seed, ms["rmse"], ms["r2"], ms["chi2"],
                mt["rmse"], mt["r2"], mt["chi2"], elapsed,
            ])

        log_progress(
            f"{label} | seed {seed:>2} : {elapsed:6.1f}s  "
            f"src R2={ms['r2']:.3f} chi2={ms['chi2']:.3f}  |  "
            f"tgt R2={mt['r2']:.3f} chi2={mt['chi2']:.3f}"
        )

log_progress(f"=== sweep complete in {(time.time()-t_sweep)/60:.1f} min ===")


## Compare against the stored seed-2 predictions

In [ ]:
import numpy as np

print("Comparing fresh seed-2 predictions against the 20-seed sweep\n" + "=" * 78)

ARRAYS = ["true_src", "mean_src", "std_src", "true_tgt", "mean_tgt", "std_tgt"]
all_identical = True
rows = []

for label in ALL_METHOD_ORDER:
    new_p, ref_p = npz_path(label, 2), ref_npz_path(label, 2)
    if not ref_p.exists():
        print(f"  {label}: NO REFERENCE at {ref_p} -- run the 20-seed sweep first")
        all_identical = False
        continue

    new_z, ref_z = np.load(new_p), np.load(ref_p)
    per_array = {}
    for k in ARRAYS:
        a, b = new_z[k], ref_z[k]
        exact = np.array_equal(a, b)
        maxdiff = float(np.abs(a.astype(np.float64) - b.astype(np.float64)).max()) if a.shape == b.shape else float("nan")
        per_array[k] = (exact, maxdiff)
        if not exact:
            all_identical = False

    worst = max(v[1] for v in per_array.values())
    every = all(v[0] for v in per_array.values())
    status = "IDENTICAL" if every else ("float-noise" if worst < 1e-6 else "DIVERGED")
    rows.append((label, status, worst))

    print(f"\n  {label}")
    for k in ARRAYS:
        exact, maxdiff = per_array[k]
        print(f"     {k:<9} bitwise={str(exact):<5}  max|diff|={maxdiff:.3e}")

    # metrics side by side -- what the difference means in reportable terms
    m_new = compute_metrics(new_z["true_tgt"], new_z["mean_tgt"], new_z["std_tgt"])
    m_ref = compute_metrics(ref_z["true_tgt"], ref_z["mean_tgt"], ref_z["std_tgt"])
    print(f"     target R2   rerun={m_new['r2']:+.6f}   sweep={m_ref['r2']:+.6f}   "
          f"delta={m_new['r2'] - m_ref['r2']:+.2e}")
    print(f"     target chi2 rerun={m_new['chi2']:.6f}   sweep={m_ref['chi2']:.6f}   "
          f"delta={m_new['chi2'] - m_ref['chi2']:+.2e}")

print("\n" + "=" * 78)
print(f"{'method':<24} {'status':>12} {'worst max|diff|':>18}")
print("-" * 78)
for label, status, worst in rows:
    print(f"{label:<24} {status:>12} {worst:>18.3e}")

print()
if all_identical:
    print("REPRODUCED: every array bitwise identical across a fresh process.")
    print("Seed 2 pins the result end to end -- the recorded seeds are portable.")
else:
    print("NOT bitwise identical -- see the per-array max|diff| above.")
    print("Values below ~1e-6 are floating-point noise (results still effectively")
    print("reproducible); anything larger means unseeded state is leaking into training.")


## Plot this run's own seed-2 result

Same combined scatter+pulls figure as the 20-seed sweep, reading from **this
notebook's own** `output/domain-shift-tng100-seed2-repro/predictions/` -- i.e. the
just-reproduced run, not the original sweep's stored copy. Since the reproduction check
above showed the two are bitwise identical, the figure is the same either way; this
just proves it renders standalone from this notebook's own artifacts.


In [ ]:
from analysis.plot_style import plot_pred_vs_true_with_uncertainty

# plot_style sets font.size=11 on import; override AFTER importing it.
plt.rcParams.update({"font.size": 18})

REPRESENTATIVE_SEED = 2  # this notebook only ever trained seed 2

seed = REPRESENTATIVE_SEED
FIT_COLOR = "#E69F00"
bin_edges = np.linspace(-4, 4, 41)
x_grid = np.linspace(-4, 4, 400)

# ── gather this seed's predictions ────────────────────────────────────────
panel = {}
for label in ALL_METHOD_ORDER:
    z = np.load(npz_path(label, seed))
    panel[label] = {
        "tgt": (z["true_tgt"], z["mean_tgt"], z["std_tgt"]),
        "src": (z["true_src"], z["mean_src"], z["std_src"]),
    }

# Common scatter limits across every scatter panel, so columns are comparable.
_all = np.concatenate([np.concatenate([panel[l][d][0], panel[l][d][1]])
                       for l in ALL_METHOD_ORDER for d in ("tgt", "src")])
s_lo, s_hi = float(_all.min()) - 0.25, float(_all.max()) + 0.25

ncol = len(ALL_METHOD_ORDER)
fig = plt.figure(figsize=(20, 16))

# Nested gridspec so the two rows WITHIN a block sit close together (they share an
# x-axis, and the upper row's tick labels are hidden), while the scatter block and
# the pull block stay visually separated. A single uniform grid cannot do both.
outer = fig.add_gridspec(2, 1, hspace=0.13, left=0.105, right=0.995, top=0.935, bottom=0.055)
gs_scatter = outer[0].subgridspec(2, ncol, hspace=0.035, wspace=0.06)
gs_pulls = outer[1].subgridspec(2, ncol, hspace=0.035, wspace=0.06)

axes = np.empty((4, ncol), dtype=object)
for r in range(2):
    for c in range(ncol):
        axes[r, c] = fig.add_subplot(gs_scatter[r, c])
        axes[2 + r, c] = fig.add_subplot(gs_pulls[r, c])

for col, label in enumerate(ALL_METHOD_ORDER):
    color = METHOD_COLORS[label]

    for block, dom in enumerate(("tgt", "src")):            # rows 0,1 = scatter
        true, mean, std = panel[label][dom]
        m = compute_metrics(true, mean, std)
        ax = axes[block, col]
        plot_pred_vs_true_with_uncertainty(
            true, mean, std, ax, color=color, markersize=3.5, annotate_fontsize=17,
            annotate=f"R$^2$={m['r2']:.3f}\nRMSE={m['rmse']:.3f}",
        )
        ax.set_xlim(s_lo, s_hi); ax.set_ylim(s_lo, s_hi)
        ax.grid(False)

    for block, dom in enumerate(("tgt", "src")):            # rows 2,3 = pulls
        true, mean, std = panel[label][dom]
        pulls = (mean - true) / std
        chi2 = float(np.mean(pulls ** 2))
        mu, sigma = norm.fit(pulls)
        ax = axes[2 + block, col]
        ax.hist(pulls, bins=bin_edges, density=True, alpha=0.85, color=color)
        ax.plot(x_grid, norm.pdf(x_grid, mu, sigma), color=FIT_COLOR, lw=1.8,
                label=f"Fit: $\\mu$={mu:.2f}, $\\sigma$={sigma:.2f}")
        ax.plot(x_grid, norm.pdf(x_grid), color="k", lw=1.3, ls="--",
                label=r"$\mathcal{N}(0,1)$")
        # legend kept a little below the 18 pt body text so the fit line fits the panel
        ax.legend(fontsize=14, frameon=False, loc="upper left", handlelength=1.0,
                  borderpad=0.12, labelspacing=0.22, borderaxespad=0.3)
        ax.text(0.97, 0.97, f"$\\chi^2$={chi2:.2f}", transform=ax.transAxes,
                va="top", ha="right", fontsize=17)
        ax.set_xlim(-4, 4); ax.set_ylim(0, 0.9); ax.grid(False)

    # Method name once, on the top row only.
    axes[0, col].set_title(label, fontsize=19, fontweight="bold", pad=10)

# ── row identity on the left column; units labelled once per 2-row block ──
axes[0, 0].set_ylabel(f"Target ({TGT_NAME})", fontsize=18)
axes[1, 0].set_ylabel(f"Source ({SRC_NAME})", fontsize=18)
axes[2, 0].set_ylabel(f"Target ({TGT_NAME})", fontsize=18)
axes[3, 0].set_ylabel(f"Source ({SRC_NAME})", fontsize=18)

for col in range(ncol):
    axes[1, col].set_xlabel(r"true $\log M_{halo}/M_\odot$", fontsize=18)
    axes[3, col].set_xlabel(r"Pull  $(\hat{\mu} - y)\,/\,\hat{\sigma}$", fontsize=18)
    axes[0, col].tick_params(labelbottom=False)   # shares x with row 1
    axes[2, col].tick_params(labelbottom=False)   # shares x with row 3
    if col > 0:
        for row in range(4):
            axes[row, col].tick_params(labelleft=False)

fig.suptitle(
    f"{SRC_NAME} $\\to$ {TGT_NAME}  --  representative seed {seed} of {len(SEEDS)}   "
    f"(scatter bars = $\\pm1\\,\\hat{{\\sigma}}$ predicted, not a 68% CI)",
    fontweight="bold", fontsize=19,
)

# Block-level y-axis labels: one per pair of rows. Positions read off the gridspec,
# which is already final (no tight_layout to shift things afterwards).
def _block_centre(rows):
    tops = [axes[r, 0].get_position().y1 for r in rows]
    bots = [axes[r, 0].get_position().y0 for r in rows]
    return (max(tops) + min(bots)) / 2.0

_x = min(axes[r, 0].get_position().x0 for r in range(4)) - 0.078
fig.text(_x, _block_centre([0, 1]), r"predicted $\log M_{halo}/M_\odot$",
         rotation="vertical", va="center", ha="center", fontsize=19)
fig.text(_x, _block_centre([2, 3]), "Density",
         rotation="vertical", va="center", ha="center", fontsize=19)

stem = OUTPUT_DIR / f"{DIRECTION}_seed{seed}_scatter_pulls"
fig.savefig(stem.with_suffix(".png"), dpi=150, bbox_inches="tight")
fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
print("saved", stem.with_suffix(".png"))
print("saved", stem.with_suffix(".pdf"))
plt.show()
